# Cybersecurity QLoRA Lab

This notebook is a thin Google Colab guide around `train_cyber_lora.py`. It does not duplicate the training implementation. The script records the unmodified Qwen2.5-7B-Instruct baseline first, runs the dataset and GPU preflight, and only then constructs LoRA layers.

No benchmark scores or training outputs are bundled here. The experiment is pending until a user runs it on an authorized dataset account and a Colab GPU. Read `BRIEF.md`, `README.md`, and `VERIFICATION_REPORT.md` before proceeding.

## 1. Put the repository in Colab

Upload or clone the repository to `/content/cybersecurity-qlora-lab`, then open this notebook from that folder. If the project is published, use its verified GitHub URL; otherwise upload the ZIP produced with this project.

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content/cybersecurity-qlora-lab')
assert (PROJECT_DIR / 'train_cyber_lora.py').exists(), 'Upload or clone the repository into /content/cybersecurity-qlora-lab first.'
%cd /content/cybersecurity-qlora-lab
print('Project files found:', len(list(PROJECT_DIR.rglob('*'))))

## 2. Verify the actual runtime GPU

The target is a T4, but Colab can assign another GPU or no GPU. The main script checks CUDA and reports free VRAM before loading the model.

In [ ]:
!nvidia-smi

## 3. Install the pinned candidate stack

Colab provides a CUDA-matched PyTorch build. The other package versions are pinned in `requirements-colab.txt`. This stack has not passed a GPU pilot in the authoring environment. If installation changes the active PyTorch/CUDA setup, restart the runtime and rerun this cell.

In [ ]:
%pip install -r requirements-colab.txt
import torch, transformers, peft, datasets, accelerate
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'transformers': transformers.__version__, 'peft': peft.__version__, 'datasets': datasets.__version__, 'accelerate': accelerate.__version__})

## 4. Mount persistent Google Drive

The default output path is `/content/drive/MyDrive/cyber-lora`. Baselines, frozen splits, checkpoints, final adapter, and results are kept there. Never put a Hugging Face token in a notebook cell or saved output.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['CYBER_LORA_OUTPUT_DIR'] = '/content/drive/MyDrive/cyber-lora'
from pathlib import Path
Path(os.environ['CYBER_LORA_OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)
print('Persistent output directory is ready.')

## 5. Hugging Face authentication and terms

Add your Hugging Face token as a Colab Secret named `HF_TOKEN` if access is required. Do not paste the token into source or notebook output. The inspected RedSage-Conv and Primus-Instruct pages require accepting gated-dataset conditions. Primus-Instruct also contains GPT-4o-generated responses; review applicable provider terms yourself. Edit `configs/experiment.yaml` only after reading and accepting relevant conditions. The default training flags are false so the pipeline stops safely without that decision.

In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    del hf_token
    print('Hugging Face authentication is configured (token value hidden).')
else:
    print('No HF_TOKEN Colab Secret found. Public metadata inspection can continue; gated training sources will remain unavailable.')

## 6. Inspect the datasets before choosing a profile

The script records live revisions, access status, configs/splits, schema, row counts where available, and the CyberMetric-500 file hash. It does not copy dataset rows into this repository. The default decision rule is T1 only when access, a documented usable license, and the scope filter all pass; otherwise T2 requires your token, accepted HF conditions, and your explicit review of teacher-output terms. Tier 3 is informational only.

In [ ]:
!python scripts/inspect_datasets.py --config configs/experiment.yaml --output "$CYBER_LORA_OUTPUT_DIR/dataset_metadata.json"

## 7. Baseline evaluation and preflight smoke test

This command prepares and freezes the training/validation data, measures CyberMetric-500 and validation loss on the original quantized base, saves the five baseline responses, then runs a one-step LoRA pilot and adapter save/reload check. It stops before full training. On the first run this includes the complete baseline and can take time. Inspect `dataset_metadata.json`, `baseline/`, and `preflight_summary.json` before the full run.

In [ ]:
!python train_cyber_lora.py --config configs/experiment.yaml --preflight-only

## 8. Full QLoRA training

Proceed only if preflight passed and the output shows a permitted dataset profile. The final record count is reduced before full training when measured throughput projects beyond the configured three-hour budget, with the decision frozen and recorded. Rerun the same command after an interruption to resume only a matching validated checkpoint.

In [ ]:
!python train_cyber_lora.py --config configs/experiment.yaml

## 9. Inspect post-training results

The script saves the adapter unmerged, performs the adapter-disabled baseline-drift check, recomputes manual validation loss on the same frozen tokens, scores the original and cyclically rotated options, and writes full row-indexed predictions. Qualitative rubric ratings remain pending until a reviewer shuffles and scores the outputs blind using `BRIEF.md` §10.

In [ ]:
import json
from pathlib import Path
out = Path(os.environ['CYBER_LORA_OUTPUT_DIR'])
results = json.loads((out / 'results.json').read_text(encoding='utf-8'))
print(json.dumps({'complete': results['complete'], 'fingerprint': results['experiment_fingerprint'], 'cybermetric': results['cybermetric_comparison'], 'validation_loss': results['validation_loss'], 'position_bias': results['position_bias']}, indent=2))

## 10. Output paths

- `baseline/`: immutable baseline components and fingerprints
- `data/`: frozen local splits and manifests (do not commit)
- `checkpoints/`: last two fingerprint-compatible checkpoints
- `final/`: final PEFT adapter
- `post_training/`: post-training loss and benchmark metrics
- `results.json`, `samples_before_after.json`, predictions, and metadata files
- `COMPLETE.json`: written only after required artifacts exist and the adapter reload/evaluation checks pass

For recovery, remount Drive, confirm the output path, and rerun the same command. Incomplete checkpoints without a matching fingerprint marker are ignored.

## Troubleshooting

- **No CUDA/GPU:** switch Colab to a GPU runtime and rerun from the GPU verification cell.
- **Gated dataset access:** accept the HF conditions while signed in, store your token in Colab Secrets, then rerun dataset inspection.
- **No eligible data profile:** review `VERIFICATION_REPORT.md`; do not turn on profile flags until license and terms decisions are documented.
- **Pilot OOM at 1,024 tokens:** the orchestrator recomputes the validation baseline at 512 before creating a new adapter. If 512 also OOMs, stop and inspect VRAM and runtime versions.
- **Drive save error:** remount Drive and verify the configured path is writable before resuming.
- **Package/API mismatch:** capture the exact versions printed by the notebook and resolve `inspect.signature` differences before changing the pinned stack. Record any change in the verification report.
- **Negative or null result:** report it unchanged. Do not tune against CyberMetric-500.